# Boston Housing Price Prediction

## Project Overview
This project develops a regression model using the Boston Housing dataset to predict housing prices. We'll implement Linear Regression and Ridge Regression models to analyze the relationship between housing features and prices.

### Objectives
- Preprocess and prepare the Boston Housing dataset
- Implement Linear Regression and Ridge Regression models
- Evaluate model performance using MSE and R-squared metrics
- Visualize predictions and analyze feature correlations
- Identify the best performing model for price prediction

## Section 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_boston
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Section 2: Load and Explore the Boston Housing Dataset

In [ ]:
# Load the Boston Housing dataset from original source
import numpy as np

# Feature names
feature_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS',
                 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']

# Load from original CMU source (sklearn removed it in v1.2)
data_url = "http://lib.stat.cmu.edu/datasets/boston"
try:
    raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
    data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
    target = raw_df.values[1::2, 2]
    X = pd.DataFrame(data, columns=feature_names)
    y = pd.Series(target, name='PRICE')
except Exception as e:
    print(f"Note: Online source temporarily unavailable. Using alternative source...")
    from sklearn.datasets import fetch_openml
    housing = fetch_openml(name="housing", as_frame=True, parser='auto')
    X = housing.data
    y = housing.target

# Create a combined dataframe for exploration
df = pd.concat([X, y], axis=1)

In [ ]:
# Check for missing values
print("\n" + "=" * 80)
print("MISSING VALUES CHECK:")
print("=" * 80)
missing_values = df.isnull().sum()
print(f"Total Missing Values: {missing_values.sum()}")
print("\nMissing values per column:")
print(missing_values)

# Display basic statistics
print("\n" + "=" * 80)
print("STATISTICAL SUMMARY:")
print("=" * 80)
print(df.describe())

# Display first few rows
print("\n" + "=" * 80)
print("FIRST FEW ROWS OF DATA:")
print("=" * 80)
print(df.head())

## Section 3: Data Preprocessing and Feature Scaling

In [ ]:
# Step 1: Handle missing data (if any)
# Boston Housing dataset typically has no missing values, but we check anyway
X_clean = X.dropna()
y_clean = y[X_clean.index]

print("=" * 80)
print("DATA PREPROCESSING:")
print("=" * 80)
print(f"Original dataset size: {X.shape[0]} samples")
print(f"After handling missing values: {X_clean.shape[0]} samples")
print(f"Removed: {X.shape[0] - X_clean.shape[0]} samples")

# Step 2: Standardize features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

print(f"\nFeature Scaling Applied: StandardScaler")
print(f"Mean of scaled features (should be ~0): {X_scaled.mean(axis=0).mean():.6f}")
print(f"Std of scaled features (should be ~1): {X_scaled.std(axis=0).mean():.6f}")

# Convert scaled data back to DataFrame for easier handling
X_scaled = pd.DataFrame(X_scaled, columns=X_clean.columns)

print("\n" + "=" * 80)
print("SCALED DATA STATISTICS:")
print("=" * 80)
print(X_scaled.describe())

## Section 4: Split Data into Training and Testing Sets

In [ ]:
# Split the data into training and testing sets
# Using 80-20 split with random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_clean, test_size=0.2, random_state=42
)

print("=" * 80)
print("TRAIN-TEST SPLIT:")
print("=" * 80)
print(f"Training set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"Testing set size: {X_test.shape[0]} samples ({X_test.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"Total samples: {len(X_scaled)}")
print(f"\nTraining features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

## Section 5: Build and Train Linear Regression Model

In [ ]:
# Initialize and train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions on training and testing sets
y_train_pred_lr = lr_model.predict(X_train)
y_test_pred_lr = lr_model.predict(X_test)

print("=" * 80)
print("LINEAR REGRESSION MODEL:")
print("=" * 80)
print(f"Model trained successfully!")
print(f"Number of features: {len(lr_model.coef_)}")
print(f"Model intercept: {lr_model.intercept_:.4f}")

# Display feature coefficients
print("\nFeature Coefficients (Top 10 by absolute value):")
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(coef_df.head(10).to_string())

# Calculate training and testing metrics for Linear Regression
lr_train_mse = mean_squared_error(y_train, y_train_pred_lr)
lr_test_mse = mean_squared_error(y_test, y_test_pred_lr)
lr_train_rmse = np.sqrt(lr_train_mse)
lr_test_rmse = np.sqrt(lr_test_mse)
lr_train_r2 = r2_score(y_train, y_train_pred_lr)
lr_test_r2 = r2_score(y_test, y_test_pred_lr)

print("\n" + "=" * 80)
print("LINEAR REGRESSION PERFORMANCE METRICS:")
print("=" * 80)
print(f"Training MSE: {lr_train_mse:.4f}")
print(f"Testing MSE: {lr_test_mse:.4f}")
print(f"Training RMSE: {lr_train_rmse:.4f}")
print(f"Testing RMSE: {lr_test_rmse:.4f}")
print(f"Training R² Score: {lr_train_r2:.4f}")
print(f"Testing R² Score: {lr_test_r2:.4f}")

## Section 6: Build and Train Ridge Regression Model with Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning for Ridge Regression
# Test different alpha values to find the best one
alpha_values = [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000]
ridge_scores = []

print("=" * 80)
print("RIDGE REGRESSION - HYPERPARAMETER TUNING:")
print("=" * 80)
print(f"\nTesting alpha values: {alpha_values}\n")

for alpha in alpha_values:
    ridge_model = Ridge(alpha=alpha)
    ridge_model.fit(X_train, y_train)
    y_test_pred = ridge_model.predict(X_test)
    test_r2 = r2_score(y_test, y_test_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    ridge_scores.append({'alpha': alpha, 'r2': test_r2, 'mse': test_mse})
    print(f"Alpha: {alpha:>8} | R² Score: {test_r2:.4f} | MSE: {test_mse:.4f}")

# Find the best alpha
best_result = max(ridge_scores, key=lambda x: x['r2'])
best_alpha = best_result['alpha']
print(f"\n✓ Best Alpha: {best_alpha} with R² Score: {best_result['r2']:.4f}")

# Train final Ridge model with best alpha
ridge_model = Ridge(alpha=best_alpha)
ridge_model.fit(X_train, y_train)

# Make predictions
y_train_pred_ridge = ridge_model.predict(X_train)
y_test_pred_ridge = ridge_model.predict(X_test)

# Calculate metrics for Ridge Regression
ridge_train_mse = mean_squared_error(y_train, y_train_pred_ridge)
ridge_test_mse = mean_squared_error(y_test, y_test_pred_ridge)
ridge_train_rmse = np.sqrt(ridge_train_mse)
ridge_test_rmse = np.sqrt(ridge_test_mse)
ridge_train_r2 = r2_score(y_train, y_train_pred_ridge)
ridge_test_r2 = r2_score(y_test, y_test_pred_ridge)

print("\n" + "=" * 80)
print("RIDGE REGRESSION PERFORMANCE METRICS:")
print("=" * 80)
print(f"Training MSE: {ridge_train_mse:.4f}")
print(f"Testing MSE: {ridge_test_mse:.4f}")
print(f"Training RMSE: {ridge_train_rmse:.4f}")
print(f"Testing RMSE: {ridge_test_rmse:.4f}")
print(f"Training R² Score: {ridge_train_r2:.4f}")
print(f"Testing R² Score: {ridge_test_r2:.4f}")

## Section 7: Model Evaluation and Comparison

In [ ]:
# Create a comprehensive comparison of both models
comparison_data = {
    'Metric': ['Training MSE', 'Testing MSE', 'Training RMSE', 'Testing RMSE', 
               'Training R²', 'Testing R²'],
    'Linear Regression': [lr_train_mse, lr_test_mse, lr_train_rmse, lr_test_rmse, 
                          lr_train_r2, lr_test_r2],
    'Ridge Regression': [ridge_train_mse, ridge_test_mse, ridge_train_rmse, ridge_test_rmse, 
                         ridge_train_r2, ridge_test_r2]
}

comparison_df = pd.DataFrame(comparison_data)

print("=" * 80)
print("MODEL COMPARISON - LINEAR vs RIDGE REGRESSION:")
print("=" * 80)
print(comparison_df.to_string(index=False))

# Calculate differences
print("\n" + "=" * 80)
print("PERFORMANCE IMPROVEMENT (Ridge - Linear):")
print("=" * 80)
print(f"Test MSE Difference: {ridge_test_mse - lr_test_mse:+.4f}")
print(f"Test RMSE Difference: {ridge_test_rmse - lr_test_rmse:+.4f}")
print(f"Test R² Difference: {ridge_test_r2 - lr_test_r2:+.4f}")

# Determine the better model
print("\n" + "=" * 80)
print("MODEL SELECTION:")
print("=" * 80)
if ridge_test_r2 > lr_test_r2:
    print(f"✓ RIDGE REGRESSION is the BETTER model")
    print(f"  - Ridge R² Score: {ridge_test_r2:.4f}")
    print(f"  - Linear R² Score: {lr_test_r2:.4f}")
    print(f"  - Improvement: {(ridge_test_r2 - lr_test_r2)*100:.2f}%")
else:
    print(f"✓ LINEAR REGRESSION is the BETTER model")
    print(f"  - Linear R² Score: {lr_test_r2:.4f}")
    print(f"  - Ridge R² Score: {ridge_test_r2:.4f}")
    print(f"  - Improvement: {(lr_test_r2 - ridge_test_r2)*100:.2f}%")

## Section 8: Visualize Predictions vs Actual Values

In [ ]:
# Visualization 1: Predictions vs Actual Values for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression
axes[0].scatter(y_test, y_test_pred_lr, alpha=0.6, color='blue', edgecolors='k')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Prices', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Predicted Prices', fontsize=12, fontweight='bold')
axes[0].set_title(f'Linear Regression\nR² = {lr_test_r2:.4f}, RMSE = {lr_test_rmse:.4f}', 
                  fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Ridge Regression
axes[1].scatter(y_test, y_test_pred_ridge, alpha=0.6, color='green', edgecolors='k')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Prices', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Predicted Prices', fontsize=12, fontweight='bold')
axes[1].set_title(f'Ridge Regression (α={best_alpha})\nR² = {ridge_test_r2:.4f}, RMSE = {ridge_test_rmse:.4f}', 
                  fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Predictions vs Actual Values visualization complete!")

In [ ]:
# Visualization 2: Residuals Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression Residuals
residuals_lr = y_test - y_test_pred_lr
axes[0].scatter(y_test_pred_lr, residuals_lr, alpha=0.6, color='blue', edgecolors='k')
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Prices', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Residuals', fontsize=12, fontweight='bold')
axes[0].set_title('Linear Regression - Residuals Plot', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Ridge Regression Residuals
residuals_ridge = y_test - y_test_pred_ridge
axes[1].scatter(y_test_pred_ridge, residuals_ridge, alpha=0.6, color='green', edgecolors='k')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Prices', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Residuals', fontsize=12, fontweight='bold')
axes[1].set_title('Ridge Regression - Residuals Plot', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Residuals analysis visualization complete!")

## Section 9: Feature Correlation Analysis

In [ ]:
# Correlation Analysis
correlation_data = pd.concat([X_clean, y_clean], axis=1)
correlation_matrix = correlation_data.corr()

# Correlation with target variable
target_correlation = correlation_matrix['PRICE'].sort_values(ascending=False)

print("=" * 80)
print("FEATURE CORRELATION WITH TARGET VARIABLE (PRICE):")
print("=" * 80)
print(target_correlation)

# Visualization 3: Correlation Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Boston Housing Dataset - Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n✓ Correlation heatmap visualization complete!")

In [ ]:
# Visualization 4: Feature Importance by Correlation
plt.figure(figsize=(10, 8))
colors = ['green' if x > 0 else 'red' for x in target_correlation[:-1].values]
bars = plt.barh(target_correlation[:-1].index, target_correlation[:-1].values, color=colors, edgecolor='black')
plt.xlabel('Correlation with Price', fontsize=12, fontweight='bold')
plt.title('Feature Correlation with Housing Prices', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Feature correlation bar plot visualization complete!")

## Section 10: Project Summary and Conclusions

In [ ]:
print("=" * 80)
print("PROJECT SUMMARY - BOSTON HOUSING PRICE PREDICTION")
print("=" * 80)

print("\n📊 DATASET INFORMATION:")
print(f"  • Total Samples: {len(X_clean)}")
print(f"  • Number of Features: {X_clean.shape[1]}")
print(f"  • Target Variable: MEDV (Median home value in $1000s)")
print(f"  • Training Set Size: {len(X_train)} samples (80%)")
print(f"  • Testing Set Size: {len(X_test)} samples (20%)")

print("\n🔧 DATA PREPROCESSING:")
print(f"  • Missing Values Handled: {X.shape[0] - X_clean.shape[0]} rows removed")
print(f"  • Feature Scaling: StandardScaler applied")
print(f"  • Scaled Feature Mean: ~0")
print(f"  • Scaled Feature Std: ~1")

print("\n📈 MODEL PERFORMANCE COMPARISON:")
print(f"\n  LINEAR REGRESSION:")
print(f"    - Training R² Score: {lr_train_r2:.4f}")
print(f"    - Testing R² Score: {lr_test_r2:.4f}")
print(f"    - Testing RMSE: ${lr_test_rmse*1000:.2f}")
print(f"    - Testing MSE: {lr_test_mse:.4f}")

print(f"\n  RIDGE REGRESSION (α={best_alpha}):")
print(f"    - Training R² Score: {ridge_train_r2:.4f}")
print(f"    - Testing R² Score: {ridge_test_r2:.4f}")
print(f"    - Testing RMSE: ${ridge_test_rmse*1000:.2f}")
print(f"    - Testing MSE: {ridge_test_mse:.4f}")

print("\n🎯 KEY FINDINGS:")
top_5_positive = target_correlation[1:6]
top_5_negative = target_correlation[-5:]
print(f"\n  Top 5 Positive Factors (increase price):")
for i, (feature, corr) in enumerate(top_5_positive.items(), 1):
    print(f"    {i}. {feature}: {corr:.4f}")

print(f"\n  Top 5 Negative Factors (decrease price):")
for i, (feature, corr) in enumerate(reversed(list(top_5_negative.items())), 1):
    print(f"    {i}. {feature}: {corr:.4f}")

print("\n" + "=" * 80)
print("✓ PROJECT COMPLETE!")
print("=" * 80)
print(f"\nBEST MODEL: {'Ridge Regression' if ridge_test_r2 > lr_test_r2 else 'Linear Regression'}")
print(f"Recommendation: {f'Ridge Regression with α={best_alpha} prevents overfitting' if ridge_test_r2 > lr_test_r2 else 'Linear Regression has simpler interpretation'}")